# 🇩🇿 DZ TOP 50 — Moteur de classement musical
**Notebook v1.0 · Projet indépendant · Algérie**

Ce notebook automatise le pipeline : collecte des charts accessibles, import de fichiers CSV, normalisation, calcul du score, historique, graphiques et exports.

> **Règle de transparence :** un classement de plateforme mesure une position, pas un nombre d'écoutes. Les données mondiales YouTube ne sont pas des écoutes algériennes. Le notebook conserve le pays et le type de mesure de chaque source et n'invente pas les données manquantes.

## Sources incluses
- **Apple Music Catalog Charts** via l'API officielle. Elle nécessite un Developer Token Apple Music (JWT) et un storefront de pays. Le code teste `dz`; si le territoire ou le chart n'est pas disponible, il le signale.
- **Spotify** : playlist éditoriale **Hot Hit Algeria** lue via sa page publique d'embed (ordre éditorial, pas un nombre d'écoutes).
- **CSV** provenant de sources autorisées (export de partenaire, relevé manuel, fournisseur de charts).
- **YouTube Data API** facultative : métadonnées et compteurs publics des vidéos, généralement mondiaux. Ce n'est pas une mesure de vues en Algérie.

Les accès aux données privées ou géographiques d'un artiste nécessitent l'autorisation du titulaire du compte/chaîne et de la plateforme.

## 0. Installation
Exécuter cette cellule une seule fois dans Google Colab ou Jupyter.

In [ ]:
%pip -q install pandas numpy requests matplotlib seaborn openpyxl

## 1. Configuration
Pour Apple Music, crée un Media ID / clé API dans Apple Developer, génère un **Developer Token JWT**, puis colle-le dans la variable ci-dessous. Ne partage jamais ta clé privée ni ton token.

In [ ]:
from pathlib import Path
from datetime import date, datetime, timedelta, timezone
import os, json, re, hashlib, warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CHART_DATE = date.today().isoformat()   # Modifier si besoin : "2026-10-30"
COUNTRY_CODE = "DZ"
TOP_N = 50

# Racine du projet : si le notebook est dans notebooks/, on remonte d'un niveau.
ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
HISTORY_DIR = DATA_DIR / "history"
EXPORT_DIR = ROOT_DIR / "exports" / CHART_DATE

for folder in [RAW_DIR, EXPORT_DIR, HISTORY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Token Apple Music (optionnel). Colle le token dans Colab Secrets ou ici temporairement.
APPLE_MUSIC_DEVELOPER_TOKEN = os.getenv("APPLE_MUSIC_DEVELOPER_TOKEN", "")
APPLE_MUSIC_STOREFRONT = "dz"

# Spotify : playlist éditoriale Algérie lue via sa page publique d'embed (sans clé API).
# Hot Hit Algeria — 50 titres, ordre éditorial Spotify (pas un classement de streams).
SPOTIFY_PLAYLIST_ID = "37i9dQZF1DX7ZqKP81pHlu"
SPOTIFY_PLAYLIST_NAME = "Hot Hit Algeria"

# YouTube Data API (optionnel) : données publiques de vidéos, pas de géolocalisation DZ.
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY", "")

print("Dossier projet :", ROOT_DIR.resolve())
print("Date du chart :", CHART_DATE)

## 2. Fonctions utilitaires et identifiants de chansons
On crée une clé stable à partir du titre et des artistes. Pour un projet réel, conserve aussi les ISRC/IDs de plateformes afin de mieux rapprocher les versions d'un même morceau.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\\s+", " ", str(value)).strip()

def make_song_id(title, artist):
    raw = (clean_text(title).lower() + "|" + clean_text(artist).lower()).encode("utf-8")
    return hashlib.sha1(raw).hexdigest()[:12]

def normalize_song_frame(df, source, country="DZ", metric="chart_position"):
    required = {"title", "artist", "rank"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes : {sorted(missing)}")
    out = df.copy()
    out["title"] = out["title"].map(clean_text)
    out["artist"] = out["artist"].map(clean_text)
    out["rank"] = pd.to_numeric(out["rank"], errors="coerce")
    out = out.dropna(subset=["title", "artist", "rank"])
    out["rank"] = out["rank"].astype(int)
    out["song_id"] = out.apply(lambda r: make_song_id(r["title"], r["artist"]), axis=1)
    out["source"] = source
    out["country"] = country
    out["metric"] = metric
    out["chart_date"] = CHART_DATE
    out["collected_at_utc"] = datetime.now(timezone.utc).isoformat()
    return out

def rank_to_points(rank, chart_size=100):
    # 1er = 100 points ; dernier du chart = 1 point.
    rank = pd.to_numeric(rank, errors="coerce")
    return np.clip((chart_size - rank + 1) / chart_size * 100, 0, 100)

## 3. Collecte Apple Music — chart Algérie
L'API officielle Apple Music expose les charts du catalogue par storefront. L'appel peut échouer si `DZ` n'est pas disponible ou si le token est invalide. Un échec est enregistré comme indisponibilité, jamais transformé en zéro.

In [ ]:
def fetch_apple_music_chart(storefront="dz", developer_token="", limit=100):
    if not developer_token:
        print("Apple Music ignoré : ajoute APPLE_MUSIC_DEVELOPER_TOKEN.")
        return pd.DataFrame()
    url = f"https://api.music.apple.com/v1/catalog/{storefront}/charts"
    headers = {"Authorization": f"Bearer {developer_token}"}
    params = {"types": "songs", "chart": "most-played", "limit": limit}
    try:
        response = requests.get(url, headers=headers, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        chart_groups = payload.get("results", {}).get("songs", [])
        if not chart_groups:
            print(f"Aucun chart songs renvoyé pour storefront={storefront}.")
            return pd.DataFrame()
        rows = []
        for group in chart_groups:
            for rank, item in enumerate(group.get("data", []), start=1):
                attrs = item.get("attributes", {})
                rows.append({
                    "title": attrs.get("name", ""),
                    "artist": attrs.get("artistName", ""),
                    "rank": rank,
                    "platform_id": item.get("id"),
                    "url": attrs.get("url", ""),
                    "genre": ", ".join(attrs.get("genreNames", [])),
                    "artwork_url": (attrs.get("artwork") or {}).get("url", ""),
                    "source_chart": group.get("name", "Apple Music Top Songs")
                })
        return normalize_song_frame(pd.DataFrame(rows), "Apple Music", storefront.upper())
    except requests.HTTPError as e:
        print("Apple Music HTTP error :", e)
        try: print(response.text[:1000])
        except Exception: pass
        return pd.DataFrame()
    except Exception as e:
        print("Erreur Apple Music :", repr(e))
        return pd.DataFrame()

apple_df = fetch_apple_music_chart(
    storefront=APPLE_MUSIC_STOREFRONT,
    developer_token=APPLE_MUSIC_DEVELOPER_TOKEN,
    limit=100
)
print("Titres Apple Music récupérés :", len(apple_df))
display(apple_df.head(10))

## 3 bis. Collecte Spotify — playlist éditoriale Algérie
Spotify ne publie pas de chart « streams par pays » accessible gratuitement. Le notebook lit donc l'ordre de la playlist éditoriale **Hot Hit Algeria** via sa page publique d'embed : c'est une vraie donnée Spotify, mais c'est un **ordre éditorial**, pas un nombre d'écoutes. La colonne `metric` vaut `playlist_order` pour cette source.

> Méthode non officielle (lecture d'une page publique). Spotify peut modifier sa page ou la bloquer. Pour un usage commercial, utilise l'API officielle avec un accès Extended Quota et l'accord de Spotify.

In [ ]:
def fetch_spotify_playlist(playlist_id, country="DZ", limit=100):
    """
    Lit une playlist Spotify publique via sa page d'embed.
    Retourne l'ordre des titres (metric="playlist_order"), pas des streams.
    """
    if not playlist_id:
        print("Spotify ignoré : SPOTIFY_PLAYLIST_ID est vide.")
        return pd.DataFrame()

    url = f"https://open.spotify.com/embed/playlist/{playlist_id}"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/130.0 Safari/537.36"
        ),
        "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
    }
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        match = re.search(
            r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>',
            response.text,
            re.S,
        )
        if not match:
            print("Spotify : __NEXT_DATA__ introuvable (page modifiée ?).")
            return pd.DataFrame()

        raw = match.group(1).strip()
        if raw.startswith("eyJ"):  # certaines variantes renvoient du base64
            import base64
            raw = base64.b64decode(raw).decode("utf-8")
        payload = json.loads(raw)
        entity = payload["props"]["pageProps"]["state"]["data"]["entity"]
        track_list = entity.get("trackList", [])
        if not track_list:
            print("Spotify : playlist vide ou non lisible.")
            return pd.DataFrame()

        rows = []
        for item in track_list:
            uri = item.get("uri", "")
            if item.get("entityType") != "track" or not uri.startswith("spotify:track:"):
                continue
            track_id = uri.split(":")[-1]
            rows.append({
                "title": item.get("title", ""),
                "artist": item.get("subtitle", ""),
                "rank": len(rows) + 1,
                "platform_id": track_id,
                "url": f"https://open.spotify.com/track/{track_id}",
                "source_chart": entity.get("name", "Spotify"),
            })

        df = pd.DataFrame(rows).head(limit)
        print(f"Spotify : {len(df)} titres lus dans « {entity.get('name')} ».")
        return normalize_song_frame(
            df, source="Spotify", country=country, metric="playlist_order"
        )
    except Exception as e:
        print("Erreur Spotify :", repr(e))
        return pd.DataFrame()


spotify_df = fetch_spotify_playlist(SPOTIFY_PLAYLIST_ID, COUNTRY_CODE)
display(spotify_df.head(10))

## 4. Importer des charts CSV d'autres plateformes
Spotify est déjà collecté automatiquement en section 3 bis. Ce bloc sert aux autres plateformes : Deezer, Anghami, Shazam, TikTok ou radios. Importe uniquement des données obtenues légalement : export partenaire, fichier fourni par un ayant droit, fournisseur sous licence ou relevé manuel d'un chart public.

Format CSV attendu : `title,artist,rank` ; colonnes facultatives : `platform_id,url,genre`.

Si tu déposes aussi `spotify_dz.csv`, il sera fusionné avec la playlist Spotify puis dédupliqué (même titre + même source).

In [ ]:
def load_chart_csv(path, source, country="DZ"):
    # utf-8-sig accepte aussi les CSV UTF-8 sans BOM ; repli latin-1 pour les exports Excel.
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
    return normalize_song_frame(df, source=source, country=country)

# Exemples : déposer les fichiers dans data/raw/
# spotify_dz.csv, deezer_dz.csv, anghami_dz.csv, shazam_dz.csv
csv_sources = {
    "Spotify": RAW_DIR / "spotify_dz.csv",
    "Deezer": RAW_DIR / "deezer_dz.csv",
    "Anghami": RAW_DIR / "anghami_dz.csv",
    "Shazam": RAW_DIR / "shazam_dz.csv",
    "TikTok": RAW_DIR / "tiktok_dz.csv",
    "Radio": RAW_DIR / "radio_dz.csv",
}

imported_frames = []
for source, path in csv_sources.items():
    if path.exists():
        try:
            imported_frames.append(load_chart_csv(path, source, "DZ"))
            print(f"{source}: importé")
        except Exception as e:
            print(f"{source}: erreur d'import - {e}")

frames = [df for df in [apple_df, spotify_df] if not df.empty] + imported_frames
raw_charts = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("Total lignes sources :", len(raw_charts))
display(raw_charts.head())

## 5. Contrôle qualité
On vérifie les champs, les doublons et le périmètre géographique. Les données `WORLD` ne sont pas admises au score DZ principal.

In [ ]:
if raw_charts.empty:
    print("Aucune donnée collectée. Ajoute un token Apple ou des CSV de charts DZ.")
    charts = pd.DataFrame(columns=[
        "title","artist","rank","song_id","source","country","metric","chart_date"
    ])
else:
    charts = raw_charts.copy()
    charts["country"] = charts["country"].str.upper()
    charts["source"] = charts["source"].astype(str)
    charts = charts.drop_duplicates(subset=["song_id","source","country","chart_date"])
    charts = charts[charts["country"].eq("DZ")].copy()

print("Lignes DZ valides :", len(charts))
print("Sources :", sorted(charts["source"].unique()) if len(charts) else [])

## 6. Calcul du DZ TOP 50
**Méthode bêta recommandée :** convertir chaque position en points, puis faire la moyenne des sources nationales réellement disponibles pour chaque titre. Cela évite de donner un zéro à une plateforme manquante.

La source Spotify correspond à l'ordre de la playlist **Hot Hit Algeria** (`metric = playlist_order`), pas à un classement de streams. Les autres sources peuvent être des positions de chart (`chart_position`). La colonne `metrics` du résultat final indique le type de mesure utilisé.

Attention : tant que le panel de sources est incomplet, publie le classement comme **DZ TOP 50 — Bêta** et affiche les plateformes couvertes.

In [ ]:
# Pondérations configurables. Ne mets une source ici que si elle fournit un chart DZ vérifiable.
SOURCE_WEIGHTS = {
    "Apple Music": 1.0,
    "Spotify": 1.0,
    "Deezer": 1.0,
    "Anghami": 1.0,
    "Shazam": 0.75,
    "TikTok": 0.50,
    "Radio": 0.75,
}

def build_dz_top50(charts_df, top_n=50):
    if charts_df.empty:
        return pd.DataFrame()
    data = charts_df.copy()
    data = data[data["country"].eq("DZ")].copy()
    data["weight"] = data["source"].map(SOURCE_WEIGHTS).fillna(0)
    data = data[data["weight"] > 0]
    if data.empty:
        return pd.DataFrame()
    data["points"] = data["rank"].apply(rank_to_points)
    data["weighted_points"] = data["points"] * data["weight"]
    grouped = data.groupby(["song_id","title","artist"], as_index=False).agg(
        weighted_total=("weighted_points","sum"),
        weight_total=("weight","sum"),
        source_count=("source","nunique"),
        sources=("source", lambda x: ", ".join(sorted(set(x)))),
        metrics=("metric", lambda x: ", ".join(sorted(set(x)))),
        best_rank=("rank","min")
    )
    grouped["score"] = grouped["weighted_total"] / grouped["weight_total"]
    grouped = grouped.sort_values(["score","source_count","best_rank"], ascending=[False,False,True])
    grouped.insert(0, "rank", range(1, len(grouped)+1))
    return grouped.head(top_n).reset_index(drop=True)

top50 = build_dz_top50(charts, TOP_N)
display(top50)

## 7. Comparaison avec la semaine précédente
Le notebook conserve chaque classement exporté. Lors d'une exécution ultérieure, il compare la position actuelle au dernier fichier historique.

In [ ]:
def load_previous_chart():
    files = sorted(HISTORY_DIR.glob("dz_top50_*.csv"))
    if not files:
        return pd.DataFrame()
    return pd.read_csv(files[-1], encoding="utf-8-sig")

previous = load_previous_chart()
if not top50.empty and not previous.empty and "song_id" in previous.columns:
    prev = previous[["song_id","rank"]].rename(columns={"rank":"previous_rank"})
    top50 = top50.merge(prev, on="song_id", how="left")
    top50["movement"] = top50["previous_rank"] - top50["rank"]
else:
    top50["previous_rank"] = np.nan
    top50["movement"] = np.nan

def movement_label(row):
    if pd.isna(row["previous_rank"]): return "NEW"
    if row["movement"] > 0: return f"+{int(row['movement'])}"
    if row["movement"] < 0: return str(int(row["movement"]))
    return "="

if not top50.empty:
    top50["movement_label"] = top50.apply(movement_label, axis=1)
display(top50.head(20))

## 8. Graphiques

In [ ]:
if not top50.empty:
    plot_df = top50.head(10).sort_values("score")
    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["title"] + " — " + plot_df["artist"], plot_df["score"])
    plt.xlabel("Score relatif (0–100)")
    plt.title("DZ TOP 10 — Score relatif")
    plt.tight_layout()
    plt.show()

    source_counts = charts.groupby("source")["song_id"].nunique().sort_values(ascending=False) if not charts.empty else pd.Series(dtype=int)
    if not source_counts.empty:
        plt.figure(figsize=(8,4))
        source_counts.plot(kind="bar")
        plt.ylabel("Nombre de titres présents")
        plt.title("Couverture par source")
        plt.tight_layout()
        plt.show()

## 9. Exports CSV et Excel
Les exports sont enregistrés dans `exports/<date>/`. Le classement est aussi archivé pour permettre les comparaisons futures.

In [ ]:
date_tag = CHART_DATE.replace("-", "")
if not top50.empty:
    csv_path = EXPORT_DIR / f"DZ_TOP_50_{date_tag}.csv"
    xlsx_path = EXPORT_DIR / f"DZ_TOP_50_{date_tag}.xlsx"
    top50.to_csv(csv_path, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        top50.to_excel(writer, sheet_name="DZ TOP 50", index=False)
        charts.to_excel(writer, sheet_name="Raw Sources", index=False)
    history_path = HISTORY_DIR / f"dz_top50_{date_tag}.csv"
    top50.to_csv(history_path, index=False, encoding="utf-8-sig")
    print("Exports créés :", csv_path, xlsx_path, history_path)
else:
    print("Pas d'export de classement : aucune donnée DZ exploitable.")

## 9 bis. Export JSON et tableau HTML
Le fichier `.json` contient le classement complet au format `records` (une entrée par titre) : il peut être ouvert dans Excel/Power BI, chargé par un site ou une API. Le fichier `.html` affiche le même tableau dans un navigateur, sans serveur.

In [ ]:
if not top50.empty:
    records = top50.astype(object).where(pd.notna(top50), None).to_dict(orient="records")

    payload = {
        "chart": "DZ TOP 50",
        "version": "beta",
        "chart_date": CHART_DATE,
        "country": COUNTRY_CODE,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "sources": sorted(charts["source"].unique().tolist()),
        "count": len(records),
        "entries": records,
    }

    json_path = EXPORT_DIR / f"DZ_TOP_50_{date_tag}.json"
    json_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

    html_path = EXPORT_DIR / f"DZ_TOP_50_{date_tag}.html"
    table_html = top50.to_html(index=False, escape=True, border=0, classes="dz-table")
    html_path.write_text(
        "<!doctype html>\n<html lang=\"fr\">\n<head>\n<meta charset=\"utf-8\">\n"
        f"<title>DZ TOP 50 — {CHART_DATE}</title>\n"
        "<style>body{font-family:system-ui,sans-serif;margin:2rem;color:#111}"
        "table{border-collapse:collapse;width:100%;font-size:14px}"
        "th,td{border:1px solid #ddd;padding:6px 10px;text-align:left}"
        "th{background:#0b6b3a;color:#fff}tr:nth-child(even){background:#f5f5f5}"
        "h1{font-size:1.4rem}</style>\n</head>\n<body>\n"
        f"<h1>🇩🇿 DZ TOP 50 — {CHART_DATE} (bêta)</h1>\n"
        f"<p>Sources : {', '.join(payload['sources']) or 'aucune'}</p>\n"
        f"{table_html}\n</body>\n</html>\n",
        encoding="utf-8",
    )

    print("Exports JSON/HTML créés :", json_path, html_path)
    display(pd.DataFrame(records).head(10))
else:
    print("Pas d'export JSON : aucune donnée DZ exploitable.")

## 10. Générer un texte prêt pour Instagram
Ce texte ne contient que les positions et les titres présents dans le classement. Relis-le avant publication.

In [ ]:
def make_instagram_caption(df, chart_date):
    if df.empty:
        return "DZ TOP 50 — édition bêta : données insuffisantes cette semaine."
    lines = [f"🇩🇿 DZ TOP 50 | {chart_date}", "", "Les hits les plus populaires en Algérie :"]
    for _, row in df.head(10).iterrows():
        move = row.get("movement_label", "")
        lines.append(f"{int(row['rank']):02d}. {row['title']} — {row['artist']} {('('+str(move)+')') if move else ''}".strip())
    lines += ["", "Classement indépendant. Méthodologie et sources : voir notre présentation."]
    return "\n".join(lines)

caption = make_instagram_caption(top50, CHART_DATE)
print(caption)
caption_path = EXPORT_DIR / f"instagram_caption_{date_tag}.txt"
caption_path.write_text(caption, encoding="utf-8")

## 11. YouTube Data API — enrichissement facultatif
Cette fonction récupère les statistiques publiques d'une vidéo connue via son ID. `viewCount` est un compteur public global ; **ne pas l'utiliser comme vues algériennes**. Pour des statistiques géographiques, le propriétaire de la chaîne doit autoriser l'accès à YouTube Analytics.

In [ ]:
def youtube_video_stats(video_ids, api_key):
    if not api_key:
        print("YouTube ignoré : ajoute YOUTUBE_API_KEY.")
        return pd.DataFrame()
    rows = []
    for start in range(0, len(video_ids), 50):
        batch = video_ids[start:start+50]
        url = "https://www.googleapis.com/youtube/v3/videos"
        params = {"part":"snippet,statistics","id":",".join(batch),"key":api_key}
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        for item in response.json().get("items", []):
            rows.append({
                "video_id": item["id"],
                "title": item["snippet"].get("title"),
                "channel": item["snippet"].get("channelTitle"),
                "published_at": item["snippet"].get("publishedAt"),
                "global_views": int(item.get("statistics", {}).get("viewCount", 0)),
                "global_likes": int(item.get("statistics", {}).get("likeCount", 0))
            })
    return pd.DataFrame(rows)

# Exemple :
# yt_stats = youtube_video_stats(["VIDEO_ID_1", "VIDEO_ID_2"], YOUTUBE_API_KEY)
# display(yt_stats)

## 12. Checklist avant publication
- [ ] Les sources et dates sont conservées.
- [ ] Les lignes utilisées sont bien des données Algérie (DZ).
- [ ] Les données mondiales sont exclues du score national.
- [ ] Les doublons et versions alternatives sont vérifiés.
- [ ] Les sources manquantes ne sont pas transformées en zéros.
- [ ] La méthode et les plateformes couvertes sont affichées.
- [ ] Le classement est relu et validé humainement.
- [ ] Les visuels et pochettes utilisés peuvent être publiés légalement.

## 13. Limites connues et évolution
1. L'API Apple Music Catalog Charts exige un Developer Token et le storefront `dz` doit être testé.
2. Spotify ne fournit pas librement un flux universel de streams par pays. Le notebook lit l'ordre de la playlist éditoriale Hot Hit Algeria via sa page d'embed : ce n'est pas un nombre d'écoutes, et cette lecture peut casser si Spotify change sa page. Depuis février 2026, l'API Web en Development Mode ne renvoie plus le contenu des playlists éditoriales.
3. Les données YouTube Data API sont des statistiques publiques globales ; les données géographiques de YouTube Analytics sont réservées aux chaînes autorisées.
4. Anghami, Deezer, Shazam, TikTok et les radios nécessitent des sources/exports autorisés et peuvent demander des accords commerciaux.
5. Les scores actuels sont des **points de classement relatifs**, pas des nombres de streams.

**Conseil de lancement :** exécute le notebook chaque semaine, archive les CSV, puis ne passe au classement public définitif qu'après plusieurs semaines de tests et une couverture de sources suffisamment stable.